# IDHM por Unidade da Federação
Análise do IDH de 1991 a 2024.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

arquivo = 'Tabela4.csv'
df = pd.read_csv(arquivo, sep=None, engine='python', encoding='latin1', header=1)
df = df.dropna(axis=1, how='all').dropna(how='all')
df.columns = [str(c).strip() for c in df.columns]

anos = [c for c in df.columns if str(c).isdigit()]
for c in anos:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '.', regex=False), errors='coerce')
    
df['Sigla'] = df['Sigla'].astype(str).str.strip()
df = df[df['Sigla'] != 'nan']
df.head()

In [ ]:
# Estados ordenados pelo maior IDH em 2024
df_ordenado = df.sort_values('2024', ascending=False)
print(df_ordenado[['Sigla', 'Estado', '2024']].to_string(index=False))

In [ ]:
# Maior melhora entre 1991 e 2024
df['Melhora'] = df['2024'] - df['1991']
maior = df.loc[df['Melhora'].idxmax()]
print(f"Maior melhora: {maior['Estado']} ({maior['Sigla']}) = {maior['Melhora']:.3f}")

# Estados em que o IDH piorou
pioraram = df[df['Melhora'] < 0]
print('\nEstados que pioraram:')
print('Nenhum estado.' if pioraram.empty else pioraram[['Sigla','Estado','1991','2024','Melhora']].to_string(index=False))

In [ ]:
# Formato largo -> formato longo
df_longo = df.melt(
    id_vars=['Sigla', 'Estado'],
    value_vars=anos,
    var_name='Ano',
    value_name='IDH'
)
df_longo['Ano'] = df_longo['Ano'].astype(int)
df_longo.head()

In [ ]:
# Apenas Minas Gerais
mg = df_longo[df_longo['Sigla'] == 'MG'].sort_values('Ano')

plt.figure(figsize=(10,5))
plt.plot(mg['Ano'], mg['IDH'], marker='o')
plt.title('Evolução do IDH de Minas Gerais')
plt.xlabel('Ano')
plt.ylabel('IDH')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Evolução do IDH de cada estado
plt.figure(figsize=(12,7))
for sigla, grupo in df_longo.groupby('Sigla'):
    grupo = grupo.sort_values('Ano')
    plt.plot(grupo['Ano'], grupo['IDH'], marker='o', markersize=3, linewidth=1.5, label=sigla)

plt.title('Evolução do IDH por estado (1991–2024)')
plt.xlabel('Ano')
plt.ylabel('IDH')
plt.ylim(0.3, 0.9)
plt.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc='upper left', title='UF')
plt.tight_layout()
plt.show()